
# Atividade de Participação – Representação Textual (PLN)

Este notebook contém **resoluções da Atividade 1:**

CPPGCC/CCN009 - PROCESSAMENTO DE LINGUAGEM NATURAL (2025.2 - T01)

Prof: Raimundo Moura

Aluno(a): Cleane Sousa Santos


In [ ]:
import re
import math
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
pd.set_option('display.max_colwidth', None)


## **Questão 1 -  Considerar a seguinte coleção de documentos:Coleção de documentos fornecida:**
1. **Certificado Digital A1 ou Certificado Digital A3: qual certificado escolher?**
2. **Certificado Digital para MEI é necessário? No mundo digital, quais as vantagens?**
3. **Certificado Digital: O que é e qual a importância no atual mundo digital?**
4. **Transferência de Veículo com Certificado Digital: Como Funciona?**
5. **Assinatura Digital Gov.br: realmente funciona? Como é?**
6. **A assinatura com certificado digital tem o mesmo valor jurídico de uma assinatura em papel.**


**i) Construir a matriz termo-documento (representação Bag-of-Words) para a coleção, usando os valores de atributos: boolean, term-frequency (9), inverse document frequency (idf) e 9-idf.**

In [ ]:
# Questão 1(i) - Representação Bag-of-Words
# Matrizes: Boolean, TF, IDF, TF-IDF

docs = [
    "Certificado Digital A1 ou Certificado Digital A3: qual certificado escolher?",
    "Certificado Digital para MEI é necessário? No mundo digital, quais as Vantagens?",
    "Certificado Digital: O que é e qual a importância no atual mundo digital?",
    "Transferência de Veículo com Certificado Digital: Como Funciona?",
    "Assinatura Digital Gov.br: realmente funciona? Como é?",
    "A assinatura com certificado digital tem o mesmo valor jurídico de uma assinatura em papel."
]

# Normalização básica
def preprocess(txt):
    txt = txt.lower()
    txt = re.sub(r"[^a-z0-9áéíóúãõç ]", " ", txt)
    return txt.strip()

tokens = [preprocess(d).split() for d in docs]
vocab = sorted({t for doc in tokens for t in doc})
N, V = len(docs), len(vocab)

# TF
tf = [[doc.count(term) for term in vocab] for doc in tokens]

# Boolean
boolean = [[1 if c > 0 else 0 for c in linha] for linha in tf]

# IDF
df = [sum(1 for linha in tf if linha[j] > 0) for j in range(V)]
idf = [math.log(N/df[j]) for j in range(V)]
idf_matrix = [[idf[j] if boolean[i][j] else 0 for j in range(V)] for i in range(N)]

# TF-IDF
tfidf = [[tf[i][j] * idf[j] for j in range(V)] for i in range(N)]

# Organização em DataFrames
idx = [f"Doc{i+1}" for i in range(N)]
matrizes = {
    "Boolean": pd.DataFrame(boolean, index=idx, columns=vocab),
    "TF": pd.DataFrame(tf, index=idx, columns=vocab),
    "IDF": pd.DataFrame(idf_matrix, index=idx, columns=vocab),
    "TF-IDF": pd.DataFrame(tfidf, index=idx, columns=vocab)
}

# Exibição em loop
for nome, matriz in matrizes.items():
    print(f"=== {nome} ===")
    display(matriz)

**ii) Pesquisar o que é um índice invetido e construi-lo para a coleção**

In [ ]:
# Questão 1(ii) — Índice Invertido
# Reaproveita 'tokens' já criado na Questão 1(i)

from collections import defaultdict

# Estrutura: termo -> lista de documentos
inverted_index = defaultdict(set)

for i, doc in enumerate(tokens, start=1):  # tokens já contém cada documento tokenizado
    for term in doc:
        inverted_index[term].add(i)

# Converto sets em listas ordenadas para exibição
inverted_index = {term: sorted(list(doc_ids)) for term, doc_ids in inverted_index.items()}

print("=== Índice invertido (termo -> docs onde aparece) ===")
for term, doc_list in list(inverted_index.items()):
    print(f"{term}: {doc_list}")

##**2. O que é o idf de um termo que ocorre em todo documento? Comparar isso com o uso de listas de stopwords.**

O valor do idf de um termo que ocorre em todos os documentos é igual a zero, pois 0 idf(t)=ln(N/N)=0. Isso significa que seu peso tf-idf também será nulo, já que qualquer frequência multiplicada por zero resulta em zero. Na prática, termos ubíquos não contribuem para diferenciar os documentos. Essa ideia se aproxima do uso de stopwords, que são removidas manualmente por serem muito comuns e pouco informativas; a diferença é que o idf realiza essa “eliminação” de forma automática, reduzindo o peso dos termos sem a necessidade de uma lista pré-definida.

In [ ]:
# Questão 2 — Exemplo com dados da questão 01 - IDF de termos que aparecem em todos os documentos

import math, pandas as pd

N = len(tokens)
vocab = sorted({t for doc in tokens for t in doc})
df = [sum(1 for doc in tokens if term in doc) for term in vocab]
idf = [math.log(N/df[j]) for j in range(len(vocab))]

df_idf = pd.DataFrame({"termo": vocab, "df": df, "idf": idf})

# Filtrar termos que aparecem em TODOS os documentos
onipresentes = df_idf[df_idf["df"] == N]
display(onipresentes)

##**3. Considerar a tabela abaixo com a frequência dos termos nos documentos: doc1, doc2 e doc3 e o valor df para cada termo t.**

| Termos | doc1 | doc2 | doc3 | df  |
|--------|------|------|------|-----|
| carro  |  27  |   4  |  24  | 500 |
| auto   |   3  |  33  |   0  | 300 |
| seguro |   0  |  33  |  29  | 100 |
| melhor |  14  |   0  |  17  | 800 |

**Para uma coleção de documentos de tamanho N = 1000, calcular os pesos 9-idf de  cada termo para cada documento.**

### Passo conceitual

A fórmula do **tf-idf** é dada por:

$$
tfidf(t,d) = tf(t,d) \cdot idf(t)
$$

em que o **idf** é calculado como:

$$
idf(t) = \ln \left( \frac{N}{df(t)} \right)
$$

onde:  
- \( tf(t,d) \) é a frequência do termo \( t \) no documento \( d \);  
- \( df(t) \) é o número de documentos que contêm o termo \( t \);  
- \( N \) é o número total de documentos na coleção.

In [ ]:
# Questão 3 — Cálculo de tf-idf

import pandas as pd
import math

# Dados fornecidos
dados = {
    "Termos": ["carro", "auto", "seguro", "melhor"],
    "doc1":   [27, 3, 0, 14],
    "doc2":   [4, 33, 33, 0],
    "doc3":   [24, 0, 29, 17],
    "df":     [500, 300, 100, 800]
}

tabela = pd.DataFrame(dados)
N = 1000  # tamanho da coleção

# Cálculo do idf
tabela["idf"] = tabela["df"].apply(lambda df: math.log(N/df))

# Cálculo do tf-idf para cada documento
for doc in ["doc1", "doc2", "doc3"]:
    tabela[f"tfidf_{doc}"] = tabela[doc] * tabela["idf"]

display(tabela[["Termos","doc1","doc2","doc3","df","idf","tfidf_doc1","tfidf_doc2","tfidf_doc3"]])

Os resultados mostram que termos menos frequentes na coleção, como “seguro”, recebem pesos mais altos, enquanto termos comuns, como “melhor”, são penalizados. Isso confirma que o TF-IDF valoriza palavras mais discriminativas.


## **4. Sobre as Word Embeddings disponíveis no site do NILC:https://nilc.icmc.usp.br/nilc/index.php/repositorio-de-word-embeddings-do-nilc, analisar os arquivos de embeddings com 100 dimensões: Word2Vec, FastText, Wang2Vec e Glove, sendo os três primeiros nos modelos CBOW e SKIP-GRAM.**
Sugestão: Analisar as embeddings das palavras Rei, Rainha, Homem e Mulher para ver se a analogia clássica é válida, ou seja, V(“rainha”) ≈ V(“rei”) - V(“homem”) + V(“mulher”)?**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from gensim.models import KeyedVectors
import math
from pathlib import Path

#  Drive
P_CSKIP_W2V = "/content/drive/My Drive/PLN - Cleane/Atividade 1/skip_s100_word2vec.txt"
P_CCBOW_W2V = "/content/drive/My Drive/PLN - Cleane/Atividade 1/cbow_s100_word2vec.txt"
P_CSKIP_W2GV = "/content/drive/My Drive/PLN - Cleane/Atividade 1/skip_s100_wang2vec.txt"
P_CCBOW_W2GV = "/content/drive/My Drive/PLN - Cleane/Atividade 1/cbow_s100_wang2vec.txt"
GLOVE_FILE = Path("/content/drive/My Drive/PLN - Cleane/Atividade 1/glove_s100.txt")
P_CSKIP_FT = "/content/drive/My Drive/PLN - Cleane/Atividade 1/skip_s100_fasttext.txt"
P_CCBOW_FT = "/content/drive/My Drive/PLN - Cleane/Atividade 1/cbow_s100_fasttext.txt"

def carregar_modelo(path_txt, is_glove=False):
    # Load models in word2vec text format.
    # Use no_header=True for GloVe files without header.
    return KeyedVectors.load_word2vec_format(path_txt, binary=False, no_header=is_glove)

def testar_analogias(model, pos, neg, topn=10):
    # Check if all words are in the vocabulary
    required_words = pos + neg
    missing = [w for w in required_words if w not in model.key_to_index]
    if missing:
        print(f"Skipping analogy test: Missing words in vocabulary - {missing}")
        return None

    # Mostra top-n e retorna índice (1-based) de 'rainha' se aparecer
    resultados = model.most_similar(positive=pos, negative=neg, topn=topn)
    print(f"\nTop {topn} para {pos} - {neg}:")
    idx_rainha = None
    for i, (pal, score) in enumerate(resultados, start=1):
        marca = "  <= RAINHA" if pal == "rainha" else ""
        print(f"{i:2d}. {pal:15s} {score:.4f}{marca}")
        if pal == "rainha" and idx_rainha is None:
            idx_rainha = i
    return idx_rainha

# Termos
POS = ["rei", "mulher"]
NEG = ["homem"]


In [ ]:
##Word2Vec
# 1) Word2Vec Skip-gram (100d)
print("=== Word2Vec — Skip-gram (100d) ===")
# Pass is_glove=False (default) for Word2Vec files
m_skip_w2v = carregar_modelo(P_CSKIP_W2V)
rank_rainha = testar_analogias(m_skip_w2v, POS, NEG, topn=10)
print("Rainha apareceu no ranking?" , "Sim, posição " + str(rank_rainha) if rank_rainha else "Não")

# 2) Word2Vec CBOW (100d)
print("\n=== Word2Vec — CBOW (100d) ===")
# Pass is_glove=False (default) for Word2Vec files
m_cbow_w2v = carregar_modelo(P_CCBOW_W2V)
rank_rainha = testar_analogias(m_cbow_w2v, POS, NEG, topn=10)
print("Rainha apareceu no ranking?" , "Sim, posição " + str(rank_rainha) if rank_rainha else "Não")

In [ ]:
# 3) FastText — Skip-gram (100d)
print("\n=== FastText — Skip-gram (100d) ===")
# Assuming FastText files might also have a header, don't pass is_glove=True
m_skip_ft = carregar_modelo(P_CSKIP_FT)
rank_rainha = testar_analogias(m_skip_ft, POS, NEG, topn=10)
print("Rainha apareceu no ranking?", "Sim, posição " + str(rank_rainha) if rank_rainha else "Não")

# 4) FastText — CBOW (100d)
print("\n=== FastText — CBOW (100d) ===")
# Assuming FastText files might also have a header, don't pass is_glove=True
m_cbow_ft = carregar_modelo(P_CCBOW_FT)
rank_rainha = testar_analogias(m_cbow_ft, POS, NEG, topn=10)
print("Rainha apareceu no ranking?", "Sim, posição " + str(rank_rainha) if rank_rainha else "Não")

In [ ]:
##Wang2Vec
# 5) Wang2Vec Skip-gram (100d)
print("\n=== Wang2Vec — Skip-gram (100d) ===")
# Pass is_glove=False (default) for Wang2Vec files
m_skip_w2gv = carregar_modelo(P_CSKIP_W2GV)
rank_rainha = testar_analogias(m_skip_w2gv, POS, NEG, topn=10)
print("Rainha apareceu no ranking?" , "Sim, posição " + str(rank_rainha) if rank_rainha else "Não")

# 6) Wang2Vec CBOW (100d)
print("\n=== Wang2Vec — CBOW (100d) ===")
# Pass is_glove=False (default) for Wang2Vec files
m_cbow_w2gv = carregar_modelo(P_CCBOW_W2GV)
rank_rainha = testar_analogias(m_cbow_w2gv, POS, NEG, topn=10)
print("Rainha apareceu no ranking?" , "Sim, posição " + str(rank_rainha) if rank_rainha else "Não")

In [ ]:
##7)Glove
print("\n=== Glove (100d) ===")
model_glove = KeyedVectors.load_word2vec_format(GLOVE_FILE, binary=False, encoding="utf-8")
res = model_glove.most_similar(positive=POS, negative=NEG, topn=10)
print("\nTop-10 resultados da analogia (GloVe):")
for i, (w, score) in enumerate(res, start=1):
    marca = "  <= RAINHA" if w == "rainha" else ""
    print(f"{i:2d}. {w:15s} {score:.4f}{marca}")


De forma geral, os modelos Skip-gram tiveram melhor desempenho na analogia do que os modelos CBOW. O GloVe apresentou vizinhanças coerentes, embora nem sempre incluísse a palavra “rainha” no top-10. Isso evidencia diferenças entre arquiteturas: CBOW tende a priorizar termos frequentes, Skip-gram captura melhor relações semânticas sutis, e o GloVe se baseia em estatísticas globais de coocorrência.


## **5. Sobre a representação de documentos usando Embeddings, descrever em uma lauda os modelos Doc2Vec e SentenceBERT. Gerar uma tabela comparativa com uma análise de custo x beneficios desses modelos**

O Doc2Vec, proposto por Le e Mikolov (2014), é uma extensão do Word2Vec voltada para documentos. Ele aprende representações vetoriais para parágrafos ou documentos inteiros adicionando um identificador de documento ao treinamento junto com as palavras do contexto. Existem duas variantes: Distributed Memory (DM), que prevê uma palavra a partir do vetor do documento e do contexto, e Distributed Bag of Words (DBOW), que prevê palavras a partir apenas do vetor do documento. O Doc2Vec é eficiente e gera vetores fixos para documentos de qualquer tamanho, preservando parte da semântica global. No entanto, apresenta limitações quanto à captura de relações complexas entre frases, pois ainda depende de um treinamento relativamente simples em comparação a modelos mais recentes baseados em transformers.

O SentenceBERT (SBERT), introduzido por Reimers e Gurevych (2019), adapta o BERT para a tarefa de representação semântica de sentenças e documentos. Ele utiliza arquitetura baseada em transformers com pooling sobre os tokens embeddings, resultando em um vetor fixo que representa a sentença ou o documento. Além disso, é treinado com técnicas de siamese networks e triplet networks, otimizando diretamente para similaridade semântica. O SentenceBERT supera modelos anteriores em tarefas como semantic textual similarity (STS), information retrieval e classificação de sentenças, pois captura relações semânticas de forma mais robusta, contextual e sensível à ordem das palavras. Sua principal desvantagem é o alto custo computacional: demanda maior memória e tempo de processamento, tanto para treinamento quanto para inferência.

Em síntese, enquanto o Doc2Vec oferece uma solução eficiente e de baixo custo para representar documentos em espaços vetoriais, adequado para grandes coleções e cenários de recursos limitados, o SentenceBERT fornece representações semânticas mais precisas e sofisticadas, apropriadas para aplicações de ponta que exigem alta qualidade na captura de similaridade semântica. A escolha entre os dois modelos deve considerar o equilíbrio entre custo computacional e benefício em termos de desempenho, de acordo com os objetivos da aplicação.

| Critério              | Doc2Vec                                   | SentenceBERT                                  |
|-----------------------|-------------------------------------------|-----------------------------------------------|
| **Arquitetura**       | Extensão do Word2Vec (DM e DBOW)          | Transformer (BERT adaptado com pooling)       |
| **Nível de repr.**    | Documento (parágrafo ou texto inteiro)    | Sentença e documento                          |
| **Captura de contexto** | Limitada (sem dependências complexas)   | Contextual e semântica profunda               |
| **Treinamento**       | Simples, requer menos dados               | Mais pesado, requer GPU e grandes datasets    |
| **Custo computacional** | Baixo                                   | Alto                                          |
| **Qualidade semântica** | Boa, mas limitada                       | Excelente, preserva nuances semânticas        |
| **Aplicações típicas** | Classificação, clustering, análise exploratória | Similaridade semântica, recuperação de informação |
| **Vantagem principal** | Eficiência e baixo custo                 | Precisão semântica superior                   |
| **Desvantagem princ.** | Perde nuances complexas                  | Elevado custo de processamento                |


## Nota de reprodutibilidade e interpretação

Este notebook apresenta uma atividade acadêmica de representação textual em PLN. Os conteúdos incluem matrizes termo-documento (BoW, TF, IDF e TF-IDF), índice invertido, comparação exploratória de embeddings de palavras (Word2Vec, FastText, Wang2Vec e GloVe), além de discussão sobre Doc2Vec e Sentence-BERT.

A IDF calculada como `log(N/df)` atribui peso zero a termos presentes em todos os documentos; isso não implica que sejam dispensáveis em qualquer tarefa. As semelhanças vetoriais e analogias dependem do corpus, do treinamento e do vocabulário. Poucos exemplos qualitativos não demonstram superioridade estatística de um método.

Os modelos pré-treinados e matrizes NumPy estão referenciados no Drive original, não neste repositório. É necessário configurar caminhos e dependências para reproduzir o trabalho. Não houve reexecução dos experimentos nesta publicação; os resultados históricos não devem ser considerados revalidados.
